# Iteration 9 — Hazard-Type Extraction with Few-Shot Flan-T5 (RQ3)

| Property | Value |
|---|---|
| **Research Question** | **RQ3** — Can large language models extract and identify underlying hazard types from process safety incidents with macro-F1 ≥ 0.7825 across multilingual datasets? |
| **Task** | Multi-class hazard-type classification (12 classes) |
| **Model** | `google/flan-t5-large` (Seq2Seq, candidate-label scoring) |
| **Label Source** | Fixed 12-class hazard taxonomy with BERTopic-assisted exploratory consolidation |
| **Input** | `TITLE + CASE_DESCRIPTION` from Process Safety incidents |
| **Data** | `master_df.json` filtered to `CASE_TYPE == "Process Safety"` |
| **Split Strategy** | `StratifiedGroupKFold` — grouped by unique text to prevent text leakage |
| **Evaluation** | Accuracy, macro-precision, macro-recall, macro-F1, weighted-F1, per-class metrics, confusion matrices |
| **Baseline Target** | Macro-F1 ≥ 0.7825 |

## Design Principles

1. **BERTopic is used only for exploratory consolidation** of noisy raw hazard labels — it does **not** define the final gold labels.
2. **A reviewed fixed hazard taxonomy** (12 classes) becomes the gold-standard label space.
3. **Flan-T5** classifies incidents via candidate-label scoring on the combined `TITLE + CASE_DESCRIPTION` text.
4. **Evaluation is performed only against reviewed gold labels**, never against raw hazard strings or BERTopic topic IDs.

## Outputs

| Type | Contents |
|---|---|
| **Artifacts** | Taxonomy JSON, BERTopic topic info, hazard mapping review template, reviewed mapping, split assignments, unique-text predictions, full predictions, error analysis |
| **Metrics** | `overall_metrics.json`, `classification_report_test.csv`, `country_metrics.csv` |
| **Figures** | Confusion matrices (raw + normalised), confidence histogram, gold vs predicted distribution, country hazard stacked bar, per-class F1, country macro-F1, precision vs recall, confidence calibration, support vs F1, top misclassifications |

## 1. Imports, Warnings, and Randomness

Standard scientific computing stack, Hugging Face Transformers for Flan-T5, scikit-learn for evaluation metrics (including multi-class ROC and PR curves), BERTopic for exploratory clustering, and matplotlib/seaborn for quality visualisations. All random seeds are fixed for reproducibility.

In [1]:
# =============================================================================
# IMPORTS, WARNINGS, RANDOMNESS
# =============================================================================
import os
import re
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Runtime Configuration

This cell configures the device, model variant, and tokenisation limits.

- **`DEBUG_MODE = True`** — uses `google/flan-t5-base` with a shorter context window for rapid prototyping.
- **`DEBUG_MODE = False`** — uses `google/flan-t5-large` with a longer context window for the final run.

The checkpoint interval controls how frequently intermediate predictions are saved to disk for restartability.

In [2]:
# =============================================================================
# DEVICE + MODEL PROFILE
# =============================================================================
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print("Using device:", DEVICE)

DEBUG_MODE = True

if DEBUG_MODE:
    MODEL_NAME = "google/flan-t5-base"
    MAX_SOURCE_TOKENS = 320
else:
    MODEL_NAME = "google/flan-t5-large"
    MAX_SOURCE_TOKENS = 384

MAX_LABEL_TOKENS = 16
CHECKPOINT_EVERY = 25

print("MODEL_NAME:", MODEL_NAME)
print("MAX_SOURCE_TOKENS:", MAX_SOURCE_TOKENS)


Using device: mps
MODEL_NAME: google/flan-t5-base
MAX_SOURCE_TOKENS: 320


## 3. Project Root Detection and Paths

The project root is discovered dynamically by searching upward from the current working directory for the `Master Dataset 34k/`, `Datasets/`, or `Results/` folders. This avoids hard-coded absolute paths and ensures reproducibility on different machines.

All output sub-directories (`artifacts/`, `metrics/`, `figures/`) are created under `Results/_iteration_9/`.

In [3]:
# =============================================================================
# PROJECT ROOT + PATHS
# =============================================================================
def find_project_root(start_path: Path, max_levels: int = 10) -> Path:
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (
            (cur / "Master Dataset 34k").exists()
            or (cur / "Datasets").exists()
            or (cur / "Results").exists()
        ):
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        "Could not find project root. Set THESIS_BASE_DIR or run the notebook from inside the repo."
    )

env_base = os.environ.get("THESIS_BASE_DIR")
if env_base:
    BASE_DIR = Path(env_base).resolve()
else:
    BASE_DIR = find_project_root(Path.cwd())

PATHS = {
    "base": BASE_DIR,
    "master_dataset_dir": BASE_DIR / "Master Dataset 34k",
    "results": BASE_DIR / "Results" / "_iteration_9",
    "artifacts": BASE_DIR / "Results" / "_iteration_9" / "artifacts",
    "metrics": BASE_DIR / "Results" / "_iteration_9" / "metrics",
    "figures": BASE_DIR / "Results" / "_iteration_9" / "figures",
}

for key, path in PATHS.items():
    if key not in {"base", "master_dataset_dir"}:
        path.mkdir(parents=True, exist_ok=True)

MASTER_JSON = PATHS["master_dataset_dir"] / "master_df.json"
FALLBACK_XLSX = BASE_DIR / "Dataset Preprocessing Step 1.xlsx"
PRED_CHECKPOINT = PATHS["artifacts"] / "unique_text_predictions_checkpoint.csv"
FINAL_PRED_FILE = PATHS["artifacts"] / "unique_text_predictions.csv"

print("BASE_DIR:", BASE_DIR)
print("MASTER_JSON exists:", MASTER_JSON.exists())
print("FALLBACK_XLSX exists:", FALLBACK_XLSX.exists())


BASE_DIR: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
MASTER_JSON exists: True
FALLBACK_XLSX exists: False


## 4. Fixed Hazard Taxonomy (12 Classes)

This is the supervised label space used for classification and evaluation. The 12 categories were derived from a combination of domain expertise, literature review, and BERTopic-assisted exploratory analysis of the raw hazard labels.

Only these 12 labels are permitted during candidate-label scoring — the model cannot invent new categories.

In [4]:
# =============================================================================
# TAXONOMY
# =============================================================================
HAZARD_TAXONOMY = [
    "Equipment Failure",
    "Leak/Spill/Release",
    "Fire/Explosion",
    "Overpressure/Process Upset",
    "Corrosion/Material Degradation",
    "Instrumentation/Control Failure",
    "Mechanical/Structural Failure",
    "Toxic Exposure/Gas Release",
    "Human/Procedure Deviation",
    "Storage/Transfer/Handling Failure",
    "Utility/System Failure",
    "Other/Unclear"
]

with open(PATHS["artifacts"] / "taxonomy.json", "w", encoding="utf-8") as f:
    json.dump(HAZARD_TAXONOMY, f, ensure_ascii=False, indent=2)

HAZARD_TAXONOMY


['Equipment Failure',
 'Leak/Spill/Release',
 'Fire/Explosion',
 'Overpressure/Process Upset',
 'Corrosion/Material Degradation',
 'Instrumentation/Control Failure',
 'Mechanical/Structural Failure',
 'Toxic Exposure/Gas Release',
 'Human/Procedure Deviation',
 'Storage/Transfer/Handling Failure',
 'Utility/System Failure',
 'Other/Unclear']

## 5. Load Data

The primary data source is `Master Dataset 34k/master_df.json`. If this file is unavailable, the notebook falls back to the preprocessed Excel file `Dataset Preprocessing Step 1.xlsx`.

In [5]:
# =============================================================================
# DATA LOADING
# =============================================================================
if MASTER_JSON.exists():
    df = pd.read_json(MASTER_JSON)
    DATA_SOURCE = str(MASTER_JSON)
elif FALLBACK_XLSX.exists():
    df = pd.read_excel(FALLBACK_XLSX)
    DATA_SOURCE = str(FALLBACK_XLSX)
else:
    raise FileNotFoundError(
        "Neither master_df.json nor Dataset Preprocessing Step 1.xlsx was found."
    )

print("Loaded from:", DATA_SOURCE)
print("Initial shape:", df.shape)
print("Columns:", sorted(df.columns.tolist())[:50])


Loaded from: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Master Dataset 34k/master_df.json
Initial shape: (34576, 36)
Columns: ['APPROVED_WITHIN_DEADLINE', 'CASENO', 'CASES_NO_OF_REGISTRATIONS', 'CASE_CLOSED_DATE', 'CASE_DESCRIPTION', 'CASE_OCCURENCE_DATE', 'CASE_SEVERITY', 'CASE_TYPE', 'COMPANY', 'COMPANY_INVOLVED_TYPE', 'COUNTRY_SHORT', 'CREATED_DATE', 'FULL_INVESTIGATION_DONE', 'FUNCTION', 'FUNCTIONAL_AREA', 'FUNCTIONAL_GROUP', 'FUNCTIONAL_LOCATION', 'FUNCTIONAL_SUB_LOCATION', 'HAZARD', 'IMM_ACTION_TAKEN_RECOM', 'LEARNINGS_ACTUAL_SEVERITY', 'LEARNINGS_POTENTIAL_RISK', 'LOCATION_SHORT', 'LOCATION_SID', 'MODIFIED_DATE', 'POTENTIAL_SEV_LEVEL', 'RISK_AREA', 'SL_COUNTRY', 'SL_LOCATION_LVL_1', 'SL_LOCATION_LVL_2', 'SL_LOCATION_LVL_3', 'SL_LOCATION_LVL_4', 'STATUS', 'TITLE', 'VALID_FROM', 'VALID_TO']


## 6. Filter to Process Safety and Select Required Columns

RQ3 targets **Process Safety** incidents exclusively. If the `CASE_TYPE` column exists, the notebook filters for `Process Safety` rows. Only columns required for hazard classification are retained: case identifier, country, date, title, description, and the raw hazard label.

In [6]:
# =============================================================================
# FILTER TO PROCESS SAFETY + COLUMN SELECTION
# =============================================================================
if "CASE_TYPE" in df.columns:
    df = df[df["CASE_TYPE"] == "Process Safety"].copy()
    print("Applied CASE_TYPE == 'Process Safety' filter.")
else:
    print("CASE_TYPE column not found. No automatic process safety filter applied.")

preferred_columns = [
    "CASENO",
    "SL_COUNTRY",
    "CASE_OCCURENCE_DATE",
    "TITLE",
    "CASE_DESCRIPTION",
    "HAZARD"
]

missing_required = [c for c in ["CASENO", "TITLE", "CASE_DESCRIPTION", "SL_COUNTRY", "HAZARD"] if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

available_columns = [c for c in preferred_columns if c in df.columns]
df = df[available_columns].copy()

print("Shape after filtering/selection:", df.shape)
df.head()


Applied CASE_TYPE == 'Process Safety' filter.
Shape after filtering/selection: (4561, 6)


,CASENO,SL_COUNTRY,CASE_OCCURENCE_DATE,TITLE,CASE_DESCRIPTION,HAZARD
8,56285,International,15/04/2025,Geborstene Entwässerung einer Hochdruckleitung...,"Im stabilen Anlagenbetrieb ist eine 2"" Entwäss...",-- Not selected --
49,55672,Netherlands,02/05/2025,Heavy lekkage in tankput,SynergiLife ter registratie van een Loss of Co...,Oil (Inc. contamination of land or water)
59,55905,Germany,12/05/2025,GT51 Ölaustritt durch fehlerhafte Schaltung de...,E-Schicht hatte kurzzeitig (ca. 20sec.) die No...,"Hazardous substance, which is non-toxic, non-C..."
79,55266,Sweden,17/04/2025,Möjligt onödig kemhantering?,Möjligt onödig kemhantering?,"Hazardous substance, which is non-toxic, non-C..."
84,55887,Sweden,18/04/2025,Kylvattenplugg släppte och kylvatten hamnade p...,Kylvattenplugg släppte och kylvatten hamnade p...,Uncontrolled release of energy


## 7. Clean Text and Build Combined Input

Whitespace is normalised and empty fields are handled. A combined `TEXT` field is constructed from `TITLE + CASE_DESCRIPTION`, which serves as the classifier input. Each unique text receives a deterministic `text_id` to prevent data leakage during splitting.

In [7]:
# =============================================================================
# CLEANING + COMBINED TEXT
# =============================================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

for col in ["TITLE", "CASE_DESCRIPTION", "HAZARD", "SL_COUNTRY"]:
    df[col] = df[col].apply(clean_text)

df["TEXT"] = (
    "Title: " + df["TITLE"].fillna("") +
    "\nDescription: " + df["CASE_DESCRIPTION"].fillna("")
).str.strip()

df = df[df["TEXT"].str.len() > 20].copy()
df = df[df["HAZARD"].str.len() > 0].copy()
df["text_id"] = pd.factorize(df["TEXT"])[0]

if "CASE_OCCURENCE_DATE" in df.columns:
    df["CASE_OCCURENCE_DATE"] = pd.to_datetime(df["CASE_OCCURENCE_DATE"], errors="coerce")

print("Shape after cleaning:", df.shape)
df.head()


Shape after cleaning: (4561, 8)


,CASENO,SL_COUNTRY,CASE_OCCURENCE_DATE,TITLE,CASE_DESCRIPTION,HAZARD,TEXT,text_id
8,56285,International,2025-04-15,Geborstene Entwässerung einer Hochdruckleitung...,"Im stabilen Anlagenbetrieb ist eine 2"" Entwäss...",-- Not selected --,Title: Geborstene Entwässerung einer Hochdruck...,0
49,55672,Netherlands,2025-05-02,Heavy lekkage in tankput,SynergiLife ter registratie van een Loss of Co...,Oil (Inc. contamination of land or water),Title: Heavy lekkage in tankput\nDescription: ...,1
59,55905,Germany,2025-05-12,GT51 Ölaustritt durch fehlerhafte Schaltung de...,E-Schicht hatte kurzzeitig (ca. 20sec.) die No...,"Hazardous substance, which is non-toxic, non-C...",Title: GT51 Ölaustritt durch fehlerhafte Schal...,2
79,55266,Sweden,2025-04-17,Möjligt onödig kemhantering?,Möjligt onödig kemhantering?,"Hazardous substance, which is non-toxic, non-C...",Title: Möjligt onödig kemhantering?\nDescripti...,3
84,55887,Sweden,2025-04-18,Kylvattenplugg släppte och kylvatten hamnade p...,Kylvattenplugg släppte och kylvatten hamnade p...,Uncontrolled release of energy,Title: Kylvattenplugg släppte och kylvatten ha...,4


## 8. Unique Texts and Raw Hazard Counts

This cell extracts the set of unique texts and computes the frequency distribution of raw hazard labels before any taxonomy consolidation. The raw counts are saved for reference and for the BERTopic exploratory step that follows.

In [8]:
# =============================================================================
# UNIQUE TEXTS + RAW HAZARD COUNTS
# =============================================================================
unique_texts = (
    df[["text_id", "TEXT", "TITLE", "CASE_DESCRIPTION"]]
    .drop_duplicates("text_id")
    .reset_index(drop=True)
)

hazard_counts = (
    df["HAZARD"]
    .value_counts(dropna=False)
    .rename_axis("raw_hazard")
    .reset_index(name="count")
)

hazard_counts.to_csv(PATHS["artifacts"] / "raw_hazard_counts.csv", index=False)

print("Rows:", len(df))
print("Unique texts:", len(unique_texts))
print("Unique raw hazards:", hazard_counts.shape[0])
hazard_counts.head(20)


Rows: 4561
Unique texts: 4525
Unique raw hazards: 190


,raw_hazard,count
0,Occupational Safety and Process Safety,1337
1,Loss of integrity,302
2,No work-related hazard,209
3,Fire / Heat,138
4,Electricity,138
5,Incorrect posture or insufficient work environ...,127
6,Sweden Hazard,105
7,-- Not selected --,100
8,Loss of control,90
9,Notification record (Eg. External regulator),87


## 9. BERTopic Exploratory Clustering on Raw Hazard Labels

This step is **exploratory only**. BERTopic clusters the noisy raw hazard strings to identify thematic groups and inform the design of the fixed 12-class taxonomy. The resulting topic assignments are **not** used as ground-truth labels for model evaluation.

A multilingual sentence transformer (`paraphrase-multilingual-MiniLM-L12-v2`) is used as the embedding backbone to handle the multilingual nature of the dataset.

In [9]:
# =============================================================================
# BERTOPIC EXPLORATORY CLUSTERING
# =============================================================================
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

hazard_docs = df["HAZARD"].astype(str).tolist()

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
    min_topic_size=10
)

topics, probs = topic_model.fit_transform(hazard_docs)
topic_info = topic_model.get_topic_info()
topic_info.to_csv(PATHS["artifacts"] / "bertopic_topic_info.csv", index=False)

bertopic_doc_topics = pd.DataFrame({
    "raw_hazard": hazard_docs,
    "topic_id": topics
})
bertopic_doc_topics.to_csv(PATHS["artifacts"] / "bertopic_doc_topics.csv", index=False)

topic_info.head(20)


W0421 08:42:34.485699 2570 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
2026-04-21 08:42:39,235 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/143 [00:00<?, ?it/s]

2026-04-21 08:42:55,042 - BERTopic - Embedding - Completed ✓
2026-04-21 08:42:55,043 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-21 08:46:20,606 - BERTopic - Dimensionality - Completed ✓
2026-04-21 08:46:20,686 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-21 08:46:25,980 - BERTopic - Cluster - Completed ✓
2026-04-21 08:46:26,205 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-04-21 08:46:27,385 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,90,-1_caustic_lighting_excessive_technical,"[caustic, lighting, excessive, technical, nois...",[Environmental noise & vibration (Inc. complai...
1,0,105,0_sweden_hazard__,"[sweden, hazard, , , , , , , , ]","[Sweden Hazard, Sweden Hazard, Sweden Hazard]"
2,1,80,1_control_loss_of_overturning,"[control, loss, of, overturning, regulator, ex...","[Loss of control, Loss of control, Loss of con..."
3,2,79,2_environmental_ecosystem_surface_other,"[environmental, ecosystem, surface, other, imp...","[Environmental, Environmental, Environmental]"
4,3,73,3_electricity___,"[electricity, , , , , , , , , ]","[Electricity, Electricity, Electricity]"
5,4,67,4_mechanical_failure_material_tested,"[mechanical, failure, material, tested, workin...","[Mechanical/Material Failure, Mechanical/Mater..."
6,5,65,5_electricity___,"[electricity, , , , , , , , , ]","[Electricity, Electricity, Electricity]"
7,6,63,6_no_related_work_hazard,"[no, related, work, hazard, sharp, edge, in, t...","[No work-related hazard, No work-related hazar..."
8,7,62,7_posture_incorrect_insufficient_environment,"[posture, incorrect, insufficient, environment...",[Incorrect posture or insufficient work enviro...
9,8,54,8_safety_occupational_process_and,"[safety, occupational, process, and, , , , , , ]","[Occupational Safety and Process Safety, Occup..."


## 10. Build Review Template for Gold Mapping

This cell generates a CSV template (`hazard_mapping_review_template.csv`) that pairs each unique raw hazard string with its BERTopic-assigned topic and a blank `final_category` column. The researcher reviews this file manually and assigns each raw hazard to exactly one of the 12 taxonomy categories.

In [10]:
# =============================================================================
# REVIEW TEMPLATE
# =============================================================================
topic_lookup = (
    bertopic_doc_topics
    .groupby("raw_hazard", as_index=False)
    .agg({"topic_id": lambda x: x.mode().iloc[0] if not x.mode().empty else -1})
)

review_template = (
    hazard_counts
    .merge(topic_lookup, on="raw_hazard", how="left")
    .sort_values(["topic_id", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

review_template["final_category"] = ""
review_template["review_notes"] = ""
review_template["review_status"] = "pending"

review_template.to_csv(PATHS["artifacts"] / "hazard_mapping_review_template.csv", index=False)

review_template.head(20)


,raw_hazard,count,topic_id,final_category,review_notes,review_status
0,Exposure to caustic substances,10,-1,,,pending
1,Lighting (Occupational: Insufficient or excess...,6,-1,,,pending
2,Toppling / collapsing structure,4,-1,,,pending
3,Environmental noise & vibration (Inc. complain...,4,-1,,,pending
4,Inadequate air ratio,4,-1,,,pending
5,Property theft,3,-1,,,pending
6,Insufficient Lighting,3,-1,,,pending
7,Pollution leading to technical breach of Envir...,2,-1,,,pending
8,Site Infrastructure theft,2,-1,,,pending
9,"Dust (Inc. complaints, technical breaches or a...",2,-1,,,pending


## 11. Manual Review Step

**Action required:** Open `Results/_iteration_9/artifacts/hazard_mapping_review_template.csv`, fill the `final_category` column using only labels from `HAZARD_TAXONOMY`, and save as `Results/_iteration_9/artifacts/hazard_mapping_reviewed.csv`.

Continue only after the reviewed mapping file exists. This step ensures that gold labels are validated by the researcher and not solely derived from automated heuristics.

In [11]:
# =============================================================================
# CHECK REVIEW FILE
# =============================================================================
mapping_file = PATHS["artifacts"] / "hazard_mapping_reviewed.csv"
print("Reviewed mapping exists:", mapping_file.exists())


Reviewed mapping exists: False


## 12. Load Reviewed Mapping and Create Gold Labels

This cell loads the manually reviewed hazard mapping (or auto-bootstraps a provisional mapping using rule-based heuristics if the reviewed file does not yet exist). Each raw hazard string is mapped to its corresponding taxonomy category, producing the `gold_hazard` column. Invalid categories are rejected, and unmapped entries default to `Other/Unclear`.

In [12]:
# =============================================================================
# LOAD REVIEWED MAPPING
# AUTO-BOOTSTRAP IF FILE IS MISSING
# =============================================================================
import re
import pandas as pd

mapping_file = PATHS["artifacts"] / "hazard_mapping_reviewed.csv"
template_file = PATHS["artifacts"] / "hazard_mapping_review_template.csv"

def bootstrap_category(raw_hazard: str) -> str:
    text = str(raw_hazard).lower().strip()

    rules = {
        "Fire/Explosion": [
            r"\bfire\b", r"\bexplos", r"\bignit", r"\bburn", r"\bflam"
        ],
        "Leak/Spill/Release": [
            r"\bleak", r"\bspill", r"\brelease", r"\bdischarge", r"\boverflow",
            r"\bescape", r"\bseep", r"\bvent", r"\bemission"
        ],
        "Overpressure/Process Upset": [
            r"\bpressure", r"\boverpressure", r"\bhigh pressure", r"\blow pressure",
            r"\btrip", r"\bprocess upset", r"\bshutdown", r"\bstartup", r"\bupset"
        ],
        "Corrosion/Material Degradation": [
            r"\bcorrosion", r"\berosion", r"\bdegradation", r"\bwear", r"\bcrack",
            r"\bfatigue"
        ],
        "Instrumentation/Control Failure": [
            r"\binstrument", r"\bcontrol", r"\bsensor", r"\balarm", r"\binterlock",
            r"\btransmitter", r"\bplc", r"\bdcs", r"\bautomation"
        ],
        "Mechanical/Structural Failure": [
            r"\bstructure", r"\bmechanical", r"\bshaft", r"\bbearing", r"\bseal",
            r"\bcasing", r"\bjoint", r"\bflange", r"\bbolt"
        ],
        "Toxic Exposure/Gas Release": [
            r"\btoxic", r"\bgas", r"\bfume", r"\bvapou?r", r"\bexposure",
            r"\binhal", r"\bchemical release"
        ],
        "Human/Procedure Deviation": [
            r"\bhuman error", r"\boperator", r"\bprocedure", r"\bpermit", r"\btraining",
            r"\bincorrect", r"\bwrong operation", r"\bdeviation"
        ],
        "Storage/Transfer/Handling Failure": [
            r"\btransfer", r"\bloading", r"\bunloading", r"\bstorage", r"\bhandling",
            r"\bcontainer", r"\bdrum", r"\bibc", r"\btank truck"
        ],
        "Utility/System Failure": [
            r"\bpower", r"\butility", r"\bsteam", r"\bcooling water", r"\bnitrogen",
            r"\bair supply", r"\belectrical", r"\bblackout"
        ],
        "Equipment Failure": [
            r"\bpump", r"\bcompressor", r"\bturbine", r"\bvalve", r"\bpipe", r"\btank",
            r"\bboiler", r"\bheat exchanger", r"\bcondenser", r"\bcooler", r"\bfailure",
            r"\bmalfunction", r"\bbreakdown", r"\brupture", r"\bburst"
        ],
    }

    for category, patterns in rules.items():
        for pattern in patterns:
            if re.search(pattern, text):
                return category

    return "Other/Unclear"

# ------------------------------------------------------------------
# If reviewed file is missing, create a provisional reviewed version
# ------------------------------------------------------------------
if not mapping_file.exists():
    if not template_file.exists():
        raise FileNotFoundError(
            f"Neither reviewed mapping nor template file exists.\n"
            f"Expected template at: {template_file}"
        )

    template_df = pd.read_csv(template_file)

    if "raw_hazard" not in template_df.columns:
        raise ValueError("Template file must contain a 'raw_hazard' column.")

    template_df["final_category"] = template_df["raw_hazard"].apply(bootstrap_category)
    if "review_notes" not in template_df.columns:
        template_df["review_notes"] = ""
    template_df["review_notes"] = template_df["review_notes"].fillna("").astype(str)
    template_df["review_notes"] = (
        template_df["review_notes"] + " AUTO-BOOTSTRAP: provisional mapping, manual review required."
    ).str.strip()
    template_df["review_status"] = "auto_bootstrap"

    template_df.to_csv(mapping_file, index=False)
    print(f"[INFO] Provisional reviewed mapping created at: {mapping_file}")

# ------------------------------------------------------------------
# Load reviewed mapping
# ------------------------------------------------------------------
mapping_df = pd.read_csv(mapping_file)

if "raw_hazard" not in mapping_df.columns or "final_category" not in mapping_df.columns:
    raise ValueError("Reviewed mapping must contain 'raw_hazard' and 'final_category'.")

invalid = sorted(set(mapping_df["final_category"].dropna()) - set(HAZARD_TAXONOMY))
if invalid:
    raise ValueError(f"Invalid categories found in reviewed mapping: {invalid}")

df = df.merge(
    mapping_df[["raw_hazard", "final_category"]].rename(
        columns={"raw_hazard": "HAZARD", "final_category": "gold_hazard"}
    ),
    on="HAZARD",
    how="left"
)

df["gold_hazard"] = df["gold_hazard"].fillna("Other/Unclear")

gold_counts = (
    df["gold_hazard"]
    .value_counts()
    .rename_axis("gold_hazard")
    .reset_index(name="count")
)
gold_counts.to_csv(PATHS["artifacts"] / "gold_hazard_counts.csv", index=False)

print("[INFO] Reviewed mapping loaded successfully.")
display(gold_counts)


[INFO] Provisional reviewed mapping created at: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_9/artifacts/hazard_mapping_reviewed.csv
[INFO] Reviewed mapping loaded successfully.


,gold_hazard,count
0,Other/Unclear,3395
1,Fire/Explosion,352
2,Toxic Exposure/Gas Release,226
3,Instrumentation/Control Failure,169
4,Human/Procedure Deviation,133
5,Mechanical/Structural Failure,69
6,Leak/Spill/Release,66
7,Overpressure/Process Upset,61
8,Equipment Failure,36
9,Utility/System Failure,29


## 13. Split Without Text Leakage

Data is split into train / validation / test partitions using `StratifiedGroupKFold`. Grouping by `text_id` ensures that the same text never appears across multiple splits. Stratification preserves the hazard-class distribution in each partition. Classes with fewer than 2 unique texts trigger an early error to prevent degenerate splits.

In [13]:
# =============================================================================
# SPLIT
# =============================================================================
unique_for_split = (
    df[["text_id", "TEXT", "gold_hazard"]]
    .drop_duplicates("text_id")
    .reset_index(drop=True)
)

min_class_count = unique_for_split["gold_hazard"].value_counts().min()
if min_class_count < 2:
    raise ValueError(
        "At least one class has fewer than 2 unique texts. Merge rare classes manually before splitting."
    )

n_splits = min(5, int(min_class_count))
if n_splits < 2:
    raise ValueError("Not enough data for stratified grouped splitting.")

sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
splits = list(
    sgkf.split(
        X=unique_for_split["TEXT"],
        y=unique_for_split["gold_hazard"],
        groups=unique_for_split["text_id"]
    )
)

train_idx, test_idx = splits[0]
train_val = unique_for_split.iloc[train_idx].copy()
test_unique = unique_for_split.iloc[test_idx].copy()

min_class_count_inner = train_val["gold_hazard"].value_counts().min()
n_splits_inner = min(5, int(min_class_count_inner)) if min_class_count_inner >= 2 else 2

sgkf_inner = StratifiedGroupKFold(n_splits=n_splits_inner, shuffle=True, random_state=SEED)
inner_splits = list(
    sgkf_inner.split(
        X=train_val["TEXT"],
        y=train_val["gold_hazard"],
        groups=train_val["text_id"]
    )
)

inner_train_idx, inner_val_idx = inner_splits[0]
train_unique = train_val.iloc[inner_train_idx].copy()
val_unique = train_val.iloc[inner_val_idx].copy()

split_assignments = pd.concat([
    train_unique.assign(split="train"),
    val_unique.assign(split="val"),
    test_unique.assign(split="test")
], ignore_index=True)

split_assignments.to_csv(PATHS["artifacts"] / "split_assignments.csv", index=False)
split_assignments["split"].value_counts()


split
train    2896
test      905
val       724
Name: count, dtype: int64

## 14. Load Flan-T5

The model is loaded in `float32` for Apple Silicon (MPS) stability. In `DEBUG_MODE`, the smaller `flan-t5-base` variant is used; for the final run, `flan-t5-large` is loaded. The model is set to evaluation mode to disable dropout.

In [14]:
# =============================================================================
# MODEL LOADING
# =============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.to(DEVICE)
model.eval()

print("Loaded model:", MODEL_NAME)


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: d81e557e-90da-49b9-9ed8-aeea197eb75d)')' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded model: google/flan-t5-base


## 15. Pre-tokenise Candidate Labels

Each of the 12 taxonomy labels is pre-tokenised once and cached. During inference, these cached token IDs are used to compute the sequence-level log-likelihood for each candidate label, avoiding redundant tokenisation.

In [15]:
# =============================================================================
# LABEL TOKEN CACHE
# =============================================================================
label_token_cache = {}

for label in HAZARD_TAXONOMY:
    label_ids = tokenizer(
        label,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LABEL_TOKENS
    ).input_ids
    label_token_cache[label] = label_ids

list(label_token_cache.keys())


['Equipment Failure',
 'Leak/Spill/Release',
 'Fire/Explosion',
 'Overpressure/Process Upset',
 'Corrosion/Material Degradation',
 'Instrumentation/Control Failure',
 'Mechanical/Structural Failure',
 'Toxic Exposure/Gas Release',
 'Human/Procedure Deviation',
 'Storage/Transfer/Handling Failure',
 'Utility/System Failure',
 'Other/Unclear']

## 16. Prompt Construction and Candidate-Label Scoring

The classification prompt instructs Flan-T5 to act as a process safety hazard classifier and return exactly one category from the allowed taxonomy. For each input text, the model scores all 12 candidate labels by computing the negative cross-entropy loss for each label sequence. Scores are converted to normalised probabilities via softmax. The label with the highest probability is selected as the prediction.

In [16]:
# =============================================================================
# PROMPT + SCORING
# =============================================================================
def build_prompt(text: str, labels: list[str]) -> str:
    label_block = "\n".join([f"- {label}" for label in labels])

    prompt = (
        "You are a process safety hazard classifier.\n"
        "Classify the incident into exactly one hazard category.\n"
        "Return only one category from the allowed list.\n\n"
        "Allowed categories:\n"
        f"{label_block}\n\n"
        f"Incident:\n{text}\n\n"
        "Answer:"
    )
    return prompt


@torch.no_grad()
def score_candidate_labels(prompt: str, labels: list[str]) -> pd.DataFrame:
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_TOKENS
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    scores = []

    for label in labels:
        label_ids = label_token_cache[label].to(DEVICE)

        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=label_ids
        )

        seq_len = label_ids.shape[1]
        sequence_log_score = -float(out.loss.detach().cpu()) * seq_len
        scores.append(sequence_log_score)

    scores = np.array(scores, dtype=np.float64)
    probs = np.exp(scores - scores.max())
    probs = probs / probs.sum()

    result = pd.DataFrame({
        "label": labels,
        "score": scores,
        "prob": probs
    }).sort_values("score", ascending=False).reset_index(drop=True)

    return result


def classify_text(text: str) -> dict:
    prompt = build_prompt(text=text, labels=HAZARD_TAXONOMY)
    scored = score_candidate_labels(prompt, HAZARD_TAXONOMY)

    top1 = scored.iloc[0]
    top2 = scored.iloc[1]

    return {
        "pred_label": top1["label"],
        "pred_confidence": float(top1["prob"]),
        "score_margin": float(top1["score"] - top2["score"]),
        "top3_labels": scored["label"].head(3).tolist(),
        "top3_probs": scored["prob"].head(3).tolist()
    }


## 17. Optional Smoke Test

A single sample is classified to verify that the prompt construction, label scoring, and probability normalisation work correctly before launching the full inference loop.

In [17]:
# =============================================================================
# SMOKE TEST
# =============================================================================
sample_text = split_assignments.iloc[0]["TEXT"]
sample_result = classify_text(sample_text)
sample_result


{'pred_label': 'Other/Unclear',
 'pred_confidence': 0.34345737841325674,
 'score_margin': 0.29549020528793335,
 'top3_labels': ['Other/Unclear',
  'Instrumentation/Control Failure',
  'Toxic Exposure/Gas Release'],
 'top3_probs': [0.34345737841325674, 0.25558954511608634, 0.18300422022074375]}

## 18. Inference on Unique Texts

Inference is performed on unique texts only (deduplicated by `text_id`) to avoid classifying the same text multiple times. Results are checkpointed to disk at regular intervals so that the run can be resumed after interruptions. MPS memory is cleared after each prediction to prevent out-of-memory errors on Apple Silicon.

In [18]:
# =============================================================================
# CHECKPOINTED INFERENCE
# =============================================================================
def clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

target_unique = split_assignments.copy()

if PRED_CHECKPOINT.exists():
    done_df = pd.read_csv(PRED_CHECKPOINT)
    done_ids = set(done_df["text_id"].tolist())
    print(f"Resuming from checkpoint. Already completed: {len(done_ids)}")
else:
    done_df = pd.DataFrame()
    done_ids = set()
    print("Starting fresh inference run.")

rows_to_run = target_unique[~target_unique["text_id"].isin(done_ids)].copy()
print("Remaining texts:", len(rows_to_run))

new_rows = []

for i, row in tqdm(rows_to_run.reset_index(drop=True).iterrows(), total=len(rows_to_run)):
    result = classify_text(row["TEXT"])

    new_rows.append({
        "text_id": row["text_id"],
        "split": row["split"],
        "gold_hazard": row["gold_hazard"],
        "pred_label": result["pred_label"],
        "pred_confidence": result["pred_confidence"],
        "score_margin": result["score_margin"],
        "top3_labels": json.dumps(result["top3_labels"]),
        "top3_probs": json.dumps(result["top3_probs"])
    })

    clear_memory()

    if (i + 1) % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.concat([done_df, pd.DataFrame(new_rows)], ignore_index=True)
        checkpoint_df.to_csv(PRED_CHECKPOINT, index=False)

final_pred = pd.concat([done_df, pd.DataFrame(new_rows)], ignore_index=True)
final_pred.to_csv(FINAL_PRED_FILE, index=False)

print("Saved:", FINAL_PRED_FILE)
print("Rows predicted:", len(final_pred))


Starting fresh inference run.
Remaining texts: 4525


  0%|          | 0/4525 [00:00<?, ?it/s]

Saved: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_9/artifacts/unique_text_predictions.csv
Rows predicted: 4525


## 19. Load Prediction File

The final (or resumed) predictions are loaded from disk. Each row contains the unique `text_id`, assigned split, gold label, predicted label, prediction confidence, score margin, and top-3 candidates.

In [19]:
# =============================================================================
# LOAD PREDICTIONS
# =============================================================================
pred_unique = pd.read_csv(FINAL_PRED_FILE)
pred_unique.head()


,text_id,split,gold_hazard,pred_label,pred_confidence,score_margin,top3_labels,top3_probs
0,2,train,Toxic Exposure/Gas Release,Other/Unclear,0.343457,0.295490,"[""Other/Unclear"", ""Instrumentation/Control Fai...","[0.34345737841325674, 0.25558954511608634, 0.1..."
1,3,train,Toxic Exposure/Gas Release,Toxic Exposure/Gas Release,0.706531,1.267073,"[""Toxic Exposure/Gas Release"", ""Other/Unclear""...","[0.7065305914937076, 0.19899778707318927, 0.04..."
2,4,train,Leak/Spill/Release,Other/Unclear,0.515942,0.352651,"[""Other/Unclear"", ""Toxic Exposure/Gas Release""...","[0.5159415213014439, 0.36261544413020336, 0.05..."
3,5,train,Other/Unclear,Other/Unclear,0.612293,1.205711,"[""Other/Unclear"", ""Leak/Spill/Release"", ""Equip...","[0.612292765922225, 0.18336887505802132, 0.101..."
4,7,train,Other/Unclear,Other/Unclear,0.679831,1.578584,"[""Other/Unclear"", ""Leak/Spill/Release"", ""Toxic...","[0.6798307739466438, 0.14022657433759092, 0.07..."


## 20. Join Predictions Back to the Full DataFrame

Unique-text predictions are joined back to the full dataframe (which may contain duplicate texts from different incidents). This produces the complete prediction file for downstream country-wise and error analyses.

In [20]:
# =============================================================================
# FULL PREDICTIONS
# =============================================================================
full_pred = df.merge(
    pred_unique[["text_id", "pred_label", "pred_confidence", "score_margin"]],
    on="text_id",
    how="left"
)

full_pred.to_csv(PATHS["artifacts"] / "full_predictions.csv", index=False)

print(full_pred.shape)
full_pred.head()


(4561, 12)


,CASENO,SL_COUNTRY,CASE_OCCURENCE_DATE,TITLE,CASE_DESCRIPTION,HAZARD,TEXT,text_id,gold_hazard,pred_label,pred_confidence,score_margin
0,56285,International,2025-04-15,Geborstene Entwässerung einer Hochdruckleitung...,"Im stabilen Anlagenbetrieb ist eine 2"" Entwäss...",-- Not selected --,Title: Geborstene Entwässerung einer Hochdruck...,0,Other/Unclear,Other/Unclear,0.669679,1.698347
1,55672,Netherlands,2025-05-02,Heavy lekkage in tankput,SynergiLife ter registratie van een Loss of Co...,Oil (Inc. contamination of land or water),Title: Heavy lekkage in tankput\nDescription: ...,1,Other/Unclear,Leak/Spill/Release,0.648951,1.692831
2,55905,Germany,2025-05-12,GT51 Ölaustritt durch fehlerhafte Schaltung de...,E-Schicht hatte kurzzeitig (ca. 20sec.) die No...,"Hazardous substance, which is non-toxic, non-C...",Title: GT51 Ölaustritt durch fehlerhafte Schal...,2,Toxic Exposure/Gas Release,Other/Unclear,0.343457,0.295490
3,55266,Sweden,2025-04-17,Möjligt onödig kemhantering?,Möjligt onödig kemhantering?,"Hazardous substance, which is non-toxic, non-C...",Title: Möjligt onödig kemhantering?\nDescripti...,3,Toxic Exposure/Gas Release,Toxic Exposure/Gas Release,0.706531,1.267073
4,55887,Sweden,2025-04-18,Kylvattenplugg släppte och kylvatten hamnade p...,Kylvattenplugg släppte och kylvatten hamnade p...,Uncontrolled release of energy,Title: Kylvattenplugg släppte och kylvatten ha...,4,Leak/Spill/Release,Other/Unclear,0.515942,0.352651


## 21. Overall Test Evaluation

This is the primary quantitative result for **RQ3**. Classification metrics (accuracy, macro-precision, macro-recall, macro-F1, weighted-F1) are computed on the held-out test split. A full per-class classification report is generated and saved.

In [21]:
# =============================================================================
# OVERALL TEST METRICS
# =============================================================================
test_pred = pred_unique[pred_unique["split"] == "test"].copy()

y_true = test_pred["gold_hazard"]
y_pred = test_pred["pred_label"]

overall_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
}

with open(PATHS["metrics"] / "overall_metrics.json", "w", encoding="utf-8") as f:
    json.dump(overall_metrics, f, indent=2)

report_df = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        labels=HAZARD_TAXONOMY,
        output_dict=True,
        zero_division=0
    )
).transpose()

report_df.to_csv(PATHS["metrics"] / "classification_report_test.csv")

overall_metrics


{'accuracy': 0.3988950276243094,
 'macro_precision': 0.1006072631072631,
 'macro_recall': 0.10457929268025422,
 'macro_f1': 0.07914572944907146,
 'weighted_f1': 0.46776553441122654}

## 22. Confusion Matrices — Raw and Normalised

Two confusion matrices are generated for the test set: a **raw count** matrix and a **row-normalised** (recall-normalised) matrix. Both use the full 12-class hazard taxonomy as axis labels. Each matrix is saved as a separate high-resolution figure (PNG + PDF, 300 dpi).

In [22]:
# =============================================================================
# CONFUSION MATRICES — RAW AND NORMALISED
# =============================================================================
cm = confusion_matrix(y_true, y_pred, labels=HAZARD_TAXONOMY)
cm_norm = confusion_matrix(y_true, y_pred, labels=HAZARD_TAXONOMY, normalize="true")

cm_df = pd.DataFrame(cm, index=HAZARD_TAXONOMY, columns=HAZARD_TAXONOMY)
cm_norm_df = pd.DataFrame(cm_norm, index=HAZARD_TAXONOMY, columns=HAZARD_TAXONOMY)

# Short labels for readability on the 12x12 matrix
SHORT_LABELS = [
    "Equip. Fail.",
    "Leak/Spill",
    "Fire/Explos.",
    "Overpress.",
    "Corrosion",
    "Instr./Ctrl",
    "Mech./Struct.",
    "Toxic Exp.",
    "Human Dev.",
    "Storage/Transf.",
    "Utility Fail.",
    "Other"
]

FIGURES_DIR = PATHS["figures"]

# ── Academic figure helpers (consistent with Iteration 8) ──
def _apply_academic_rcparams():
    matplotlib.rcParams.update({
        'font.family': 'serif',
        'font.serif':  ['Times New Roman', 'DejaVu Serif'],
        'font.size':   13,
    })

def _restore_rcparams():
    matplotlib.rcParams.update(matplotlib.rcParamsDefault)

def _save_academic_figure(fig, base_name):
    png_path = FIGURES_DIR / f'{base_name}.png'
    pdf_path = FIGURES_DIR / f'{base_name}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    print(f"  [OK] Saved: {base_name}.png + .pdf")
    plt.close(fig)

# --- Raw Confusion Matrix ---
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_df,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.8,
    linecolor="white",
    xticklabels=SHORT_LABELS,
    yticklabels=SHORT_LABELS,
    annot_kws={"size": 8, "fontweight": "bold"},
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_xlabel("Predicted Label", fontsize=16)
ax.set_ylabel("True Label", fontsize=16)
ax.set_title("Confusion Matrix — Test Set (Raw Counts)", fontsize=18, fontweight="bold", pad=14)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
fig.tight_layout()
_save_academic_figure(fig, "confusion_matrix_test")
_restore_rcparams()

# --- Normalised Confusion Matrix ---
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm_df,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.8,
    linecolor="white",
    xticklabels=SHORT_LABELS,
    yticklabels=SHORT_LABELS,
    annot_kws={"size": 8, "fontweight": "bold"},
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_xlabel("Predicted Label", fontsize=16)
ax.set_ylabel("True Label", fontsize=16)
ax.set_title("Normalised Confusion Matrix — Test Set (Row-Normalised)", fontsize=18, fontweight="bold", pad=14)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
fig.tight_layout()
_save_academic_figure(fig, "confusion_matrix_test_normalized")
_restore_rcparams()

print("Confusion matrices saved to:", PATHS["figures"])

  [OK] Saved: confusion_matrix_test.png + .pdf
  [OK] Saved: confusion_matrix_test_normalized.png + .pdf
Confusion matrices saved to: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_9/figures


## 23. Country-wise Evaluation

Per-country classification metrics are computed for countries with at least 20 test samples. This enables analysis of geographic variation in model performance and supports the multilingual aspect of RQ3.

In [23]:
# =============================================================================
# COUNTRY METRICS
# =============================================================================
test_full = full_pred.merge(
    split_assignments[["text_id", "split"]],
    on="text_id",
    how="left"
)
test_full = test_full[test_full["split"] == "test"].copy()

country_rows = []
min_country_samples = 20

for country, g in test_full.groupby("SL_COUNTRY"):
    if len(g) < min_country_samples:
        continue

    yt = g["gold_hazard"]
    yp = g["pred_label"]

    country_rows.append({
        "SL_COUNTRY": country,
        "n_samples": len(g),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "accuracy": accuracy_score(yt, yp)
    })

country_metrics = pd.DataFrame(country_rows).sort_values("macro_f1", ascending=False)
country_metrics.to_csv(PATHS["metrics"] / "country_metrics.csv", index=False)

country_metrics.head(20)


,SL_COUNTRY,n_samples,macro_precision,macro_recall,macro_f1,accuracy
3,UK,272,0.122071,0.142845,0.094496,0.352941
0,Germany,368,0.103936,0.100610,0.092652,0.470109
1,Netherlands,115,0.092682,0.094933,0.083557,0.486957
2,Sweden,139,0.109554,0.144965,0.054003,0.287770


## 24. Confidence Distribution and Class Distribution Plots

Three separate figures are produced:
1. **Prediction confidence histogram** — distribution of the model's prediction confidence scores on the test set.
2. **Gold vs Predicted hazard distribution** — grouped bar chart comparing the true and predicted class frequencies.
3. **Predicted hazard categories by country** — stacked bar chart showing how predicted hazard types are distributed across countries.

All figures are saved individually at 300 dpi in both PNG and PDF formats.

In [24]:
# =============================================================================
# DISTRIBUTION PLOTS 
# =============================================================================

# --- 1. Prediction Confidence Histogram ---
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(
    test_pred["pred_confidence"],
    bins=30,
    color="#08306b",
    edgecolor="white",
    linewidth=0.8,
    alpha=0.9,
)
ax.set_xlabel("Prediction Confidence", fontsize=16)
ax.set_ylabel("Count", fontsize=16)
ax.set_title("Prediction Confidence Distribution — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, "confidence_histogram")
_restore_rcparams()

# --- 2. Gold vs Predicted Hazard Distribution ---
gold_dist = test_pred["gold_hazard"].value_counts().reindex(HAZARD_TAXONOMY, fill_value=0)
pred_dist = test_pred["pred_label"].value_counts().reindex(HAZARD_TAXONOMY, fill_value=0)

x_pos = np.arange(len(HAZARD_TAXONOMY))
bar_width = 0.35

_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(x_pos - bar_width / 2, gold_dist.values, bar_width, label="Gold", color="#08306b", edgecolor="white")
ax.bar(x_pos + bar_width / 2, pred_dist.values, bar_width, label="Predicted", color="#2171b5", edgecolor="white")
ax.set_xticks(x_pos)
ax.set_xticklabels(SHORT_LABELS, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Count", fontsize=16)
ax.set_title("Gold vs Predicted Hazard Distribution — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.legend(frameon=True, fontsize=13)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, "hazard_distribution_gold_vs_pred")
_restore_rcparams()

# --- 3. Predicted Hazard Categories by Country (Stacked Bar) ---
country_hazard = (
    test_full.groupby(["SL_COUNTRY", "pred_label"])
    .size()
    .reset_index(name="count")
)

pivot = country_hazard.pivot_table(
    index="SL_COUNTRY", columns="pred_label", values="count", fill_value=0
)
pivot = pivot.reindex(columns=HAZARD_TAXONOMY, fill_value=0)

_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
pivot.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    colormap="tab20",
    edgecolor="white",
    linewidth=0.3,
)
ax.set_xlabel("Country", fontsize=16)
ax.set_ylabel("Count", fontsize=16)
ax.set_title("Predicted Hazard Categories by Country — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.legend(
    title="Hazard Category",
    fontsize=6,
    title_fontsize=8,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True,
)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", labelsize=13)
ax.grid(axis="y", alpha=0.3, linestyle="--")
fig.tight_layout()
_save_academic_figure(fig, "country_hazard_stacked")
_restore_rcparams()

print("Distribution plots saved.")

  [OK] Saved: confidence_histogram.png + .pdf
  [OK] Saved: hazard_distribution_gold_vs_pred.png + .pdf
  [OK] Saved: country_hazard_stacked.png + .pdf
Distribution plots saved.


## 25. Additional Result Visualisations

Six additional diagnostic figures are generated:
1. **Per-class F1 bar chart** — horizontal bar chart of F1 scores for each hazard category.
2. **Country-wise macro-F1 bar chart** — horizontal bar chart comparing macro-F1 across countries.
3. **Per-class precision vs recall** — paired horizontal bar chart.
4. **Confidence calibration plot** — binned accuracy vs predicted confidence with a perfect-calibration diagonal.
5. **Class support vs F1 scatter** — shows whether model performance correlates with class frequency.
6. **Top misclassification pairs** — the 15 most frequent gold → predicted error pairs.

All figures are saved individually at 300 dpi in both PNG and PDF formats.

In [25]:
# =============================================================================
# ADDITIONAL RESULT VISUALISATIONS
# =============================================================================
per_class = report_df.loc[
    report_df.index.isin(HAZARD_TAXONOMY)
].copy()
per_class = per_class.sort_values("f1-score", ascending=True)

# Map full taxonomy names to short labels for plotting
label_map = dict(zip(HAZARD_TAXONOMY, SHORT_LABELS))
per_class_short = per_class.copy()
per_class_short.index = per_class_short.index.map(label_map)

# =============================================================================
# 1. Per-Class F1 Bar Chart
# =============================================================================
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    per_class_short.index,
    per_class_short["f1-score"],
    color="#08306b",
    edgecolor="white",
    height=0.6,
)
ax.set_xlabel("F1 Score", fontsize=16)
ax.set_xlim(0, 1)
ax.set_title("Per-Class F1 Score — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
for i, v in enumerate(per_class_short["f1-score"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)
fig.tight_layout()
_save_academic_figure(fig, "per_class_f1")
_restore_rcparams()

# =============================================================================
# 2. Country-wise Macro F1 Bar Chart
# =============================================================================
cm_sorted = country_metrics.sort_values("macro_f1", ascending=True)

_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    cm_sorted["SL_COUNTRY"],
    cm_sorted["macro_f1"],
    color="#08306b",
    edgecolor="white",
    height=0.6,
)
ax.set_xlabel("Macro F1", fontsize=16)
ax.set_xlim(0, 1)
ax.set_title("Country-wise Macro F1 — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
for i, (_, row) in enumerate(cm_sorted.iterrows()):
    ax.text(row["macro_f1"] + 0.01, i, f"{row['macro_f1']:.3f}", va="center", fontsize=8)
fig.tight_layout()
_save_academic_figure(fig, "country_macro_f1")
_restore_rcparams()

# =============================================================================
# 3. Per-Class Precision vs Recall Paired Bars
# =============================================================================
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
y_pos = np.arange(len(per_class_short))
bar_h = 0.35

ax.barh(y_pos - bar_h / 2, per_class_short["precision"], bar_h,
        label="Precision", color="#08306b", edgecolor="white")
ax.barh(y_pos + bar_h / 2, per_class_short["recall"], bar_h,
        label="Recall", color="#2171b5", edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels(per_class_short.index, fontsize=8)
ax.set_xlabel("Score", fontsize=16)
ax.set_xlim(0, 1)
ax.set_title("Per-Class Precision vs Recall — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.legend(frameon=True, fontsize=13)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, "precision_vs_recall")
_restore_rcparams()

# =============================================================================
# 4. Confidence Calibration (Binned Accuracy vs Confidence)
# =============================================================================
cal_df = test_pred.copy()
cal_df["correct"] = (cal_df["gold_hazard"] == cal_df["pred_label"]).astype(int)
cal_df["conf_bin"] = pd.cut(cal_df["pred_confidence"], bins=10)

cal_grouped = (
    cal_df.groupby("conf_bin", observed=True)
    .agg(
        mean_confidence=("pred_confidence", "mean"),
        accuracy=("correct", "mean"),
        count=("correct", "size"),
    )
    .reset_index()
)

_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(
    cal_grouped["mean_confidence"],
    cal_grouped["accuracy"],
    width=0.08,
    color="#08306b",
    edgecolor="white",
    alpha=0.9,
)
ax.plot([0, 1], [0, 1], "r--", linewidth=1.5, label="Perfect Calibration")
ax.set_xlabel("Mean Predicted Confidence", fontsize=16)
ax.set_ylabel("Accuracy in Bin", fontsize=16)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Confidence Calibration — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.legend(frameon=True, fontsize=13)
ax.grid(alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, "confidence_calibration")
_restore_rcparams()

# =============================================================================
# 5. Class Support vs F1 Scatter
# =============================================================================
support_f1 = per_class[["support", "f1-score"]].copy()
support_f1["category"] = [label_map.get(c, c) for c in support_f1.index]

_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    support_f1["support"],
    support_f1["f1-score"],
    color="#08306b",
    s=80,
    edgecolors="white",
    linewidth=0.8,
    zorder=3,
)
for _, row in support_f1.iterrows():
    ax.annotate(
        row["category"],
        (row["support"], row["f1-score"]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=7,
    )
ax.set_xlabel("Number of Test Samples (Support)", fontsize=16)
ax.set_ylabel("F1 Score", fontsize=16)
ax.set_ylim(0, 1)
ax.set_title("Class Support vs F1 Score — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.grid(alpha=0.3, linestyle="--")
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, "support_vs_f1")
_restore_rcparams()

# =============================================================================
# 6. Top Misclassification Pairs
# =============================================================================
errors = test_pred[test_pred["gold_hazard"] != test_pred["pred_label"]].copy()
errors["pair"] = errors["gold_hazard"] + " → " + errors["pred_label"]

top_errors = (
    errors["pair"]
    .value_counts()
    .head(15)
    .reset_index()
)
top_errors.columns = ["misclassification", "count"]

_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    top_errors["misclassification"][::-1],
    top_errors["count"][::-1],
    color="#08306b",
    edgecolor="white",
    height=0.6,
)
ax.set_xlabel("Count", fontsize=16)
ax.set_title("Top 15 Misclassification Pairs — Test Set", fontsize=18, fontweight="bold", pad=14)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.tick_params(axis="y", labelsize=7)
ax.tick_params(axis="x", labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, "top_misclassifications")
_restore_rcparams()

print("All additional plots saved.")

  [OK] Saved: per_class_f1.png + .pdf
  [OK] Saved: country_macro_f1.png + .pdf
  [OK] Saved: precision_vs_recall.png + .pdf
  [OK] Saved: confidence_calibration.png + .pdf
  [OK] Saved: support_vs_f1.png + .pdf
  [OK] Saved: top_misclassifications.png + .pdf
All additional plots saved.


## 26. Error Analysis File

High-confidence misclassifications are exported for manual review. These cases — where the model was confident but wrong — are particularly informative for the discussion section and for identifying systematic failure modes.

In [26]:
# =============================================================================
# ERROR ANALYSIS
# =============================================================================
error_analysis = test_full[test_full["gold_hazard"] != test_full["pred_label"]].copy()

cols = [
    "CASENO", "SL_COUNTRY", "TITLE", "CASE_DESCRIPTION",
    "HAZARD", "gold_hazard", "pred_label", "pred_confidence", "score_margin"
]
cols = [c for c in cols if c in error_analysis.columns]

error_analysis = error_analysis[cols].sort_values(
    ["pred_confidence", "score_margin"],
    ascending=[False, False]
)

error_analysis.to_csv(PATHS["artifacts"] / "error_analysis.csv", index=False)

error_analysis.head(30)


,CASENO,SL_COUNTRY,TITLE,CASE_DESCRIPTION,HAZARD,gold_hazard,pred_label,pred_confidence,score_margin
2374,9408,UK,Glycol Regeneration Unit 30 Low Pressure Trip,Glycol Regeneration Unit 30 (GRU) tripped on L...,Build up of an explosive atmosphere,Fire/Explosion,Overpressure/Process Upset,0.998775,7.844730
489,38374,Sweden,Oskarshamn 3 - Oil leak in pipe conector of tu...,Oskarshamn 3 - Oil leak in pipe conector of tu...,Occupational Safety and Process Safety,Other/Unclear,Leak/Spill/Release,0.998435,7.104181
4262,6388,UK,IP feedwater leak from pipework,During walk round a leak was found on the IP/L...,Water,Other/Unclear,Leak/Spill/Release,0.998411,6.806152
848,34569,UK,Leakage from seal oil system,Unit 3 was on outage. Generator was in Hydroge...,Hazardous Substance Chemical / Materials,Other/Unclear,Leak/Spill/Release,0.997916,6.887220
956,30052,UK,PSNH - U1 C16 Oil Burner hose leak,"@ Approx 07:30 Sunday morning 04/12/22, a fuel...",Occupational Safety and Process Safety,Other/Unclear,Leak/Spill/Release,0.997014,6.713838
1519,20826,UK,WTP Effluent Tank Leak in to Bund.,Shift team received a radio message from Kaefe...,Exposure to caustic substances,Toxic Exposure/Gas Release,Leak/Spill/Release,0.996849,6.553328
1342,4391,UK,Feed pump oil leak.,Oil reported leaking from the feed pump. Leak ...,Contamination of water,Other/Unclear,Leak/Spill/Release,0.996693,6.121199
2534,9294,UK,U8 ST HP Stop V/V PO Drain Line Fracture,Unit was at full load to a PN when the lube oi...,Loss of Containment,Other/Unclear,Leak/Spill/Release,0.996021,6.834395
3918,57933,UK,U6 Condenser Seawater Leak,"During pre-start checks, it was noticed that t...",Loss of containment,Other/Unclear,Leak/Spill/Release,0.995955,5.734146
565,20012,UK,Water treatment Plant Effluent Leak,Significant effluent leak from under leak pad ...,"Hazardous substance, which is non-toxic, non-C...",Toxic Exposure/Gas Release,Leak/Spill/Release,0.995780,6.101027


## 27. Final Summary

All outputs are saved under `Results/_iteration_9/`.

### Artifacts
| File | Description |
|---|---|
| `taxonomy.json` | Fixed 12-class hazard taxonomy |
| `raw_hazard_counts.csv` | Frequency distribution of raw hazard labels before consolidation |
| `bertopic_topic_info.csv` | BERTopic topic summary (exploratory only) |
| `bertopic_doc_topics.csv` | Per-document BERTopic topic assignments |
| `hazard_mapping_review_template.csv` | Template for manual raw → taxonomy mapping |
| `hazard_mapping_reviewed.csv` | Reviewed (or auto-bootstrapped) mapping |
| `gold_hazard_counts.csv` | Gold-label frequency distribution |
| `split_assignments.csv` | Train / validation / test split assignments |
| `unique_text_predictions.csv` | Per-unique-text Flan-T5 predictions |
| `full_predictions.csv` | Full dataframe with predictions joined back |
| `error_analysis.csv` | High-confidence misclassifications for manual review |

### Metrics
| File | Description |
|---|---|
| `overall_metrics.json` | Accuracy, macro-precision, macro-recall, macro-F1, weighted-F1 |
| `classification_report_test.csv` | Full per-class classification report |
| `country_metrics.csv` | Per-country macro-F1, precision, recall, accuracy |

### Figures (PNG + PDF, 300 dpi)
| File | Description |
|---|---|
| `confusion_matrix_test` | Raw-count confusion matrix (12 × 12) |
| `confusion_matrix_test_normalized` | Row-normalised confusion matrix |
| `confidence_histogram` | Prediction confidence distribution |
| `hazard_distribution_gold_vs_pred` | Gold vs predicted class frequencies |
| `country_hazard_stacked` | Predicted hazard categories by country |
| `per_class_f1` | Per-class F1 horizontal bar chart |
| `country_macro_f1` | Country-wise macro-F1 horizontal bar chart |
| `precision_vs_recall` | Per-class precision vs recall paired bars |
| `confidence_calibration` | Binned accuracy vs confidence calibration plot |
| `support_vs_f1` | Class support vs F1 score scatter |
| `top_misclassifications` | Top 15 misclassification pairs |

In [27]:
print("Iteration 9 pipeline complete.")
print("Results directory:", PATHS["results"])


Iteration 9 pipeline complete.
Results directory: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_9
